# Module 04: Data Structures Deep Dive

Learn how Python's data structures work under the hood and when to use each one in ML projects.

**Focus areas:**
- List internals and amortized O(1) append
- Dict internals (hash tables)
- Set operations for ML
- Tuple performance
- Collections module (Counter, defaultdict, deque, OrderedDict)
- heapq and bisect
- Performance analysis for large datasets

## 1. Lists: Dynamic Arrays Under the Hood

Python lists are **dynamic arrays**, not linked lists. Elements are stored contiguously in memory.

When the array fills up, Python allocates ~1.125x more space and copies everything over. This makes most appends O(1) but occasionally O(n) — hence **amortized O(1)**.

In [ ]:
import sys
import time

# See how list capacity grows
data = []
for i in range(20):
    data.append(i)
    size = sys.getsizeof(data)
    if i < 10 or i >= 15:
        print("len =", len(data), "| size =", size, "bytes | overhead per elem:", (size - 56) / max(1, len(data)))

print("\n--- Append vs Insert Performance ---")

# Append is O(1) amortized
lst = []
t0 = time.time()
for i in range(100000):
    lst.append(i)
t_append = time.time() - t0

# Insert at beginning is O(n)
lst = []
t0 = time.time()
for i in range(100000):
    lst.insert(0, i)
t_insert = time.time() - t0

print("Append 100k (O(1)*):", round(t_append, 4), "sec")
print("Insert at 0 (O(n)):", round(t_insert, 4), "sec")
print("Insert is", round(t_insert / t_append, 1), "x slower")

## 2. Dictionaries: Hash Tables

Dicts use hash tables. When you look up `d[key]`, Python:
1. Computes `hash(key)`
2. Maps hash to an index
3. Checks if that slot has the right key
4. Returns the value

This is O(1) average, O(n) worst case (hash collisions).

In [ ]:
import time
import random

print("--- Dict vs List Lookup Performance ---")

random.seed(42)
n = 100000

# Build data
keys = list(range(n))
values = [random.random() for _ in range(n)]

# Build dict and parallel list
d = dict(zip(keys, values))
list_keys = keys
list_values = values

# Search in dict (O(1))
search_keys = [random.randint(0, n-1) for _ in range(10000)]

t0 = time.time()
for k in search_keys:
    _ = d.get(k, None)
t_dict = time.time() - t0
print("Dict lookup 10k items (O(1)):", round(t_dict, 5), "sec")

# Search in list (O(n))
t0 = time.time()
for k in search_keys[:100]:  # Only 100 to keep time reasonable
    try:
        idx = list_keys.index(k)
        _ = list_values[idx]
    except ValueError:
        pass
t_list = time.time() - t0
print("List lookup 100 items (O(n)):", round(t_list, 5), "sec")
print("\nDict is dramatically faster for lookups!")

## 3. Sets: Unordered Unique Collections

Sets implement the same hash table as dicts but store only keys. Perfect for membership testing and deduplication.

In [ ]:
import time

print("--- Set vs List Membership ---")

n = 100000
data_list = list(range(n))
data_set = set(data_list)

# Test membership
test_items = [-1, 0, n//2, n-1, n+1]

t0 = time.time()
for item in test_items:
    _ = item in data_list
t_list = time.time() - t0
print("List membership (5 items):", round(t_list, 6), "sec")

t0 = time.time()
for item in test_items:
    _ = item in data_set
t_set = time.time() - t0
print("Set membership (5 items):", round(t_set, 6), "sec")

print("\n--- Set operations useful for ML ---")

# ML scenario: check data leakage between train/test
train_ids = set(range(0, 800))
test_ids = set(range(700, 1000))

leakage = train_ids & test_ids
print("Leaked samples (in both train and test):", sorted(leakage)[:5], "..." if len(leakage) > 5 else "")
print("Leakage count:", len(leakage))

# Deduplication
duplicate_ids = [1, 2, 2, 3, 3, 3, 4, 5, 5]
unique = set(duplicate_ids)
print("\nDedup:", duplicate_ids, "->", sorted(unique))

## 4. Tuples: Immutable & Hashable

Tuples are like lists but **immutable**. This allows them to be used as dictionary keys.

In [ ]:
import time

print("--- Tuple vs List Performance ---")

# Creating
t0 = time.time()
for _ in range(1000000):
    _ = (1, 2, 3, 4, 5)
t_tuple_create = time.time() - t0

t0 = time.time()
for _ in range(1000000):
    _ = [1, 2, 3, 4, 5]
t_list_create = time.time() - t0

print("Create 1M tuples:", round(t_tuple_create, 4), "sec")
print("Create 1M lists:", round(t_list_create, 4), "sec")

# Tuples as dict keys
coord_data = {}
coord_data[(0.5, 0.3)] = "point_A"
coord_data[(0.1, 0.9)] = "point_B"
print("\nTuple as dict key:", coord_data)

# ML: storing feature vectors as tuples when you need them hashable
feature_cache = {}
sample = (0.2, 0.5, 0.8)
feature_cache[sample] = "class_1"
print("Cached feature:", feature_cache.get((0.2, 0.5, 0.8)))

## 5. Counter: Counting Made Simple

`Counter` is a dict subclass designed for counting hashable objects.

In [ ]:
from collections import Counter

print("--- Counter for ML Data Analysis ---")

# Simulate class labels from a dataset
labels = ["cat", "dog", "cat", "bird", "dog", "cat", "dog", "dog", "cat", "bird"]
counts = Counter(labels)

print("Class distribution:", dict(counts))
print("Most common:", counts.most_common(2))

# Detecting class imbalance
total = sum(counts.values())
for cls, cnt in counts.items():
    pct = cnt / total * 100
    print("  " + cls + ":", cnt, "(" + str(round(pct, 1)) + "%)")

min_class = min(counts.values())
max_class = max(counts.values())
if min_class / max_class < 0.3:
    print("WARNING: Severe class imbalance detected!")

# Counter operations
more_labels = ["cat", "fish", "dog"]
counts.update(more_labels)
print("\nAfter update:", dict(counts))

# Top-N frequent items
n = 2
top_n = counts.most_common(n)
print("Top", n, "classes:", top_n)

## 6. defaultdict: Auto-Initializing Dicts

`defaultdict` avoids `KeyError` by calling a factory function for missing keys.

In [ ]:
from collections import defaultdict

print("--- defaultdict for Data Grouping ---")

# Group samples by class label
samples = [
    ("cat", [0.5, 0.2]),
    ("dog", [0.8, 0.3]),
    ("cat", [0.6, 0.1]),
    ("bird", [0.9, 0.7]),
    ("dog", [0.7, 0.4])
]

groups = defaultdict(list)
for label, features in samples:
    groups[label].append(features)

for label, feats in groups.items():
    print(label + ":", feats)

# Nested defaultdict for hierarchical data
metrics = defaultdict(lambda: defaultdict(list))
metrics["model_a"]["loss"].append(0.5)
metrics["model_a"]["loss"].append(0.4)
metrics["model_b"]["loss"].append(0.6)
print("\nNested metrics:", dict(metrics))

# Without defaultdict (clunky)
groups_manual = {}
for label, features in samples:
    if label not in groups_manual:
        groups_manual[label] = []
    groups_manual[label].append(features)
print("\nManual grouping (more code):", dict(groups_manual))

## 7. deque: Double-Ended Queue for Sliding Windows

`deque` provides O(1) append/pop from both ends. Perfect for sliding windows in time series.

In [ ]:
from collections import deque
import time

print("--- deque for Sliding Window ---")

# Simulate training loss tracking
window_size = 5
loss_window = deque(maxlen=window_size)

# Simulate 20 training epochs
import random
random.seed(42)
for epoch in range(20):
    loss = random.random() * (1.0 - epoch * 0.03)  # decreasing trend
    loss_window.append(loss)
    avg_loss = sum(loss_window) / len(loss_window)
    print("Epoch", epoch, "| loss:", round(loss, 4), "| window avg:", round(avg_loss, 4), "| window:", list(loss_window))

print("\n--- deque vs list for left-pop ---")

# deque O(1) pop left
dq = deque(range(100000))
t0 = time.time()
for _ in range(10000):
    dq.popleft()
t_deque = time.time() - t0
print("deque popleft 10k times:", round(t_deque, 5), "sec")

# list O(n) pop(0)
lst = list(range(100000))
t0 = time.time()
for _ in range(10000):
    lst.pop(0)
t_list = time.time() - t0
print("list pop(0) 10k times:", round(t_list, 5), "sec")
print("deque is", round(t_list / t_deque, 1), "x faster")

## 8. heapq: Priority Queues for Top-K Problems

`heapq` implements a heap queue algorithm. Use it for finding top-k items without sorting everything.

In [ ]:
import heapq
import time
import random

print("--- heapq.nlargest for Feature Selection ---")

# ML: Top-k feature importance
feature_importance = {
    "age": 0.45,
    "income": 0.82,
    "education": 0.31,
    "occupation": 0.28,
    "location": 0.65,
    "credit_score": 0.71,
    "loan_amount": 0.55
}

k = 3
top_features = heapq.nlargest(k, feature_importance, key=feature_importance.get)
print("Top", k, "features:", top_features)
for feat in top_features:
    print("  " + feat + ":", feature_importance[feat])

print("\n--- Performance: heapq vs sort for top-10 ---")

n = 1000000
data = [random.random() for _ in range(n)]

# heapq.nlargest
t0 = time.time()
top10_heap = heapq.nlargest(10, data)
t_heap = time.time() - t0

# Full sort
t0 = time.time()
top10_sort = sorted(data, reverse=True)[:10]
t_sort = time.time() - t0

print("heapq.nlargest(10):", round(t_heap, 4), "sec")
print("sorted()[:10]:", round(t_sort, 4), "sec")
print("heapq is", round(t_sort / t_heap, 1), "x faster")

## 9. bisect: Binary Search on Sorted Lists

`bisect` finds insertion points in sorted lists using binary search — O(log n).

In [ ]:
import bisect

print("--- bisect for Threshold-Based Operations ---")

# ML: Find probability bin for calibration
bins = [0.1, 0.3, 0.5, 0.7, 0.9]
probabilities = [0.05, 0.25, 0.45, 0.65, 0.85, 0.95]

for p in probabilities:
    bin_idx = bisect.bisect_left(bins, p)
    print("Prob", p, "-> bin index", bin_idx, "(bin range:", bins[bin_idx-1] if bin_idx > 0 else 0.0, "-", bins[bin_idx] if bin_idx < len(bins) else 1.0, ")")

print("\n--- Maintaining a Sorted List ---")

# Efficiently insert into sorted list
sorted_scores = [0.1, 0.3, 0.5, 0.7, 0.9]
new_score = 0.45
pos = bisect.bisect_left(sorted_scores, new_score)
print("Insert", new_score, "at position", pos)
sorted_scores.insert(pos, new_score)
print("Updated list:", sorted_scores)

print("\n--- Performance: bisect vs list.index ---")
import time

large_sorted = list(range(0, 100000, 2))  # Even numbers only
target = 54321

t0 = time.time()
pos = bisect.bisect_left(large_sorted, target)
found = large_sorted[pos] == target if pos < len(large_sorted) else False
t_bisect = time.time() - t0
print("bisect search (O(log n)):", round(t_bisect, 7), "sec, found =", found)

t0 = time.time()
try:
    idx = large_sorted.index(target)
    found = True
except ValueError:
    found = False
t_index = time.time() - t0
print("list.index (O(n)):", round(t_index, 7), "sec, found =", found)

## 10. Data Structure Selection Guide

| Task | Right Choice | Wrong Choice | Why |
|------|-------------|-------------|-----|
| Check if item exists | `set` | `list` | O(1) vs O(n) |
| Count frequencies | `Counter` | Manual dict | Less code, more features |
| Group by key | `defaultdict` | Manual dict | No boilerplate |
| Sliding window | `deque(maxlen=N)` | `list` | O(1) push/pop both ends |
| Top-K items | `heapq.nlargest` | `sorted()[:k]` | O(n log k) vs O(n log n) |
| Sorted lookup | `bisect` | `list.index` | O(log n) vs O(n) |
| Fixed record | `tuple` | `list` | Hashable, immutable |
| Ordered mapping | `dict` (3.7+) | `OrderedDict` | Regular dict preserves order |

**Key takeaway:** The right data structure can make your ML code 100-1000x faster. Always ask "what am I doing with this data?" before choosing.